In [48]:
import pandas as pd

In [49]:
df = pd.read_csv("./anime_data_cleaned.csv")
df.shape

(30440, 22)

In [50]:
class DSU:
    def __init__(self, ids: list[int]):
        self.n = len(ids)
        self.rank = {i: 1 for i in ids}
        self.parent = {i: i for i in ids}

    def find(self, mal_id:int):
        if mal_id != self.parent[mal_id]:
            self.parent[mal_id ] = self.find(self.parent[mal_id])

        return self.parent[mal_id]

    def union(self, x: int, y: int) -> bool:
        p1, p2 = self.find(x), self.find(y)

        if p1 == p2:
            return False

        self.n -= 1
        if self.rank[p1] > self.rank[p2]:
            self.parent[p2] = p1
            self.rank[p1] += self.rank[p2]
        else:
            self.parent[p1] = p2
            self.rank[p2] += self.rank[p1]

        return True


In [51]:
ids = df["id"]
dsu = DSU(ids.tolist())

In [52]:
for idx, relation_list in df["related_anime"].dropna().items():
    for relation_str in relation_list.split(";"):
        relation_str = relation_str.strip()
        splitted = relation_str.split("|")
        if len(splitted) != 3:
            continue
        id_, title, relation = splitted
        try:
            dsu.union(ids[idx], int(id_))
        except KeyError:
            continue

print(dsu.n)

15743


In [53]:
df["franchise_id"] = df["id"].apply(dsu.find)
df["franchise_id"].unique()

array([   85,  4106,   264, ..., 64310, 64311, 64313], shape=(15743,))

In [56]:
df[df["title"].str.contains("Shingeki")]["franchise_id"].value_counts()

franchise_id
25777    23
34642     4
85        1
Name: count, dtype: int64

In [57]:
df.to_csv("anime_data_cleaned.csv", index=False, encoding="utf-8")